# DSAI 490 - Assignment 1

This notebook trains and compares three models on the required image dataset:

- Convolutional Autoencoder (AE)
- Denoising Autoencoder
- Variational Autoencoder (VAE)

The reusable code lives under `src/representation_learning/`, while this notebook focuses on running the experiments and generating the report figures.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Running outside Colab. Google Drive was not mounted.')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    fallback_root = Path('/content/assignment1')
    if (fallback_root / 'src').exists():
        PROJECT_ROOT = fallback_root

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from representation_learning import (
    ExperimentConfig,
    build_autoencoder,
    build_datasets,
    build_denoising_autoencoder,
    build_vae,
    evaluate_reconstruction,
    interpolate_latent,
    plot_dataset_samples,
    plot_generated_samples,
    plot_interpolation_grid,
    plot_latent_projection,
    plot_model_reconstructions,
    plot_reconstruction_grid,
    plot_training_curves,
    plot_vae_losses,
    project_latent,
    sample_vae,
    train_model,
)

tf.keras.utils.set_random_seed(42)

config = ExperimentConfig(
    drive_data_root='/content/drive/MyDrive/PUT_YOUR_DATASET_PATH_HERE',
    extract_root='/content/datasets/dsai490_assignment1',
    output_dir='/content/dsai490_outputs',
    batch_size=32,
    seed=42,
    latent_dim=16,
    ae_epochs=30,
    vae_epochs=40,
    learning_rate=1e-3,
    noise_std=0.15,
)

config


In [ ]:
train_ds, val_ds, test_ds, metadata = build_datasets(config)
print(metadata.to_dict())
plot_dataset_samples(train_ds, metadata, rows=2, columns=5, title='Dataset Preview')


## 1. Autoencoder

In [ ]:
ae_encoder, ae_decoder, autoencoder = build_autoencoder(config)
ae_history = train_model(autoencoder, train_ds, val_ds, config)
plot_training_curves(ae_history, title='Autoencoder Training Curves')
ae_metrics, ae_examples = evaluate_reconstruction(autoencoder, test_ds, metadata)
plot_reconstruction_grid(ae_examples, title='Autoencoder Reconstructions')
ae_metrics


## 2. Denoising Autoencoder

In [ ]:
denoise_encoder, denoise_decoder, denoising_autoencoder = build_denoising_autoencoder(config)
denoise_history = train_model(denoising_autoencoder, train_ds, val_ds, config)
plot_training_curves(denoise_history, title='Denoising Autoencoder Training Curves')
denoise_metrics, denoise_examples = evaluate_reconstruction(denoising_autoencoder, test_ds, metadata)
plot_reconstruction_grid(denoise_examples, title='Denoising Autoencoder Results')
denoise_metrics


## 3. Variational Autoencoder

In [ ]:
vae_encoder, vae_decoder, vae = build_vae(config)
vae_history = train_model(vae, train_ds, val_ds, config)
plot_vae_losses(vae_history, title='VAE Loss Curves')
vae_metrics, vae_examples = evaluate_reconstruction(vae, test_ds, metadata)
plot_reconstruction_grid(vae_examples, title='VAE Reconstructions')
vae_metrics


## 4. AE vs VAE Reconstruction Comparison

In [ ]:
plot_model_reconstructions(
    ae_examples['original'],
    ae_examples['reconstructed'],
    vae_examples['reconstructed'],
    title='AE vs VAE Reconstructions',
    max_items=6,
)


## 5. Latent Space Visualization

In [ ]:
ae_points_2d, labels = project_latent(ae_encoder, test_ds, method='pca', dims=2)
plot_latent_projection(ae_points_2d, labels, metadata.label_names, dims=2, title='AE Latent Space (2D PCA)')

vae_points_2d, labels = project_latent(vae_encoder, test_ds, method='pca', dims=2)
plot_latent_projection(vae_points_2d, labels, metadata.label_names, dims=2, title='VAE Latent Space (2D PCA)')

vae_points_3d, labels = project_latent(vae_encoder, test_ds, method='pca', dims=3)
plot_latent_projection(vae_points_3d, labels, metadata.label_names, dims=3, title='VAE Latent Space (3D PCA)')


## 6. VAE Sample Generation

In [ ]:
generated_samples = sample_vae(
    decoder=vae_decoder,
    num_samples=10,
    latent_dim=config.latent_dim,
    seed=config.seed,
)
plot_generated_samples(generated_samples, title='VAE Generated Samples', columns=5)


## 7. Latent Interpolation

In [ ]:
test_batch = next(iter(test_ds))
interpolated_images = interpolate_latent(
    encoder=vae_encoder,
    decoder=vae_decoder,
    image_a=test_batch['image'][0].numpy(),
    image_b=test_batch['image'][1].numpy(),
    steps=8,
)
plot_interpolation_grid(interpolated_images, title='VAE Latent Interpolation')


## 8. Results Table

In [ ]:
results_table = pd.DataFrame(
    [
        {'model': 'Autoencoder', **ae_metrics},
        {'model': 'Denoising Autoencoder', **denoise_metrics},
        {'model': 'Variational Autoencoder', **vae_metrics},
    ]
)
results_table
